<a href="https://colab.research.google.com/github/durfishans/Build-Document-Publish-/blob/main/CRM_LEAD_QUALIFIER_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [3]:
try:
  client= OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
except Exception as e:
  print(f"Error initializing OpenAI client:{e}")
  print("Please ensure your OpenAI_APIKEY is set in your envirment variables.")



In [4]:
def lookup_domain_info(domain: str)->str:
  print(f"-> Tool acticated: Looking for domain info for {domain}...")

  mock_data = {
      "acmecorp.com": {"industry":"SAAS", "size":"500-1000 employees","revenue":"$50M-$100M"},
      "widgetcorp.com": {"industry":"Manufacturing", "size":"100-500 employees","revenue":"$25M-$50M"},
      "globalfin.com":{"industry":"Finance", "size":"5000+ employees","revenue":"$1B+"}
  }

  info = mock_data.get(domain, {"industry":"Unknown", "size":"N/A", "revenue":"N/A"})

  return json.dumps(info)

In [5]:
def check_crm_history(email: str)-> str:
  print(f"-> Tool activated: Looking for CRM history for {email}....")

  mock_data = {
      "dur@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
      "bob@widgetcorp.com" : {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
      "default" : {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."}
  }

  history = mock_data.get(email, mock_data["default"])
  return json.dumps(history)

In [6]:
history_check = check_crm_history('james.iredell@examplepetstore.com')
print(history_check)

-> Tool activated: Looking for CRM history for james.iredell@examplepetstore.com....
{"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."}


In [7]:
def calculate_lead_score(data_summary: str)->str:
  print(f"->Tool activated: Calculating lead score...")

  data = json.loads(data_summary)
  print(data)
  score = "Low"  #Default score

  if data["domain_info"].get("revenue", "").startswith("$1B"):
    score = "High"
  elif data["crm_history"].get("status", "").startswith("Active Opportunity"):
    score = "High"
  elif data["domain_info"].get("revenue", "").startswith("$50M"):
    score = "Medium"

  return json.dumps({"lead_score":score})

In [8]:
domain_info_output = lookup_domain_info('acmecorp.com')
crm_history_output = check_crm_history('dur@acmecorp.com')

# Parse the JSON strings into Python dictionaries
domain_data = json.loads(domain_info_output)
crm_data = json.loads(crm_history_output)

# Combine the dictionaries
combined_data = {
    "domain_info": domain_data,
    "crm_history": crm_data
}

# Convert the combined dictionary back to a JSON string
combined_json_string = json.dumps(combined_data)

print(combined_json_string)

-> Tool acticated: Looking for domain info for acmecorp.com...
-> Tool activated: Looking for CRM history for dur@acmecorp.com....
{"domain_info": {"industry": "SAAS", "size": "500-1000 employees", "revenue": "$50M-$100M"}, "crm_history": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."}}


Now you can use this `combined_json_string` as the `data_summary` argument for the `calculate_lead_score` function.

In [14]:
lead_score = calculate_lead_score(combined_json_string)
print(lead_score)

->Tool activated: Calculating lead score...
{'domain_info': {'industry': 'SAAS', 'size': '500-1000 employees', 'revenue': '$50M-$100M'}, 'crm_history': {'last_contact': '2025-11-15', 'status': 'Cold Lead', 'notes': 'Attended webinar, no follow-up yet.'}}
{"lead_score": "Medium"}


In [9]:
AVAILABLE_FUNCTIONS = {
    "lookup_domain_info" : lookup_domain_info,
    "check_crm_history" : check_crm_history,
    "calculate_lead_score" : calculate_lead_score
}

In [15]:
tools_schema = [
    {
        "type" : "function",
        "function": {
            "name" : "lookup_domain_info",
            "description" : "Retreives domain information about the company it's size, revenue etc.",
            "parameters" : {
                "type" : "object",
                "properties" : {
                    "domain" : {"type": "string", "description" : "The company's domain name, eg 'acme.corp' "},
                },
                "required" : ["domain"],
            },
        },
    },
    {
        "type" : "function",
        "function": {
            "name" : "check_crm_history",
            "description" : "Checks the internal CRM system for past contacts, status and notes for a specific contact",
            "parameters" : {
                "type" : "object",
                "properties" : {
                    "email" : {"type": "string", "description" : "The contact's email address, eg 'dur@acmecorp.com' "},
                },
                "required" : ["email"],
            },
        },

    },
    {
        "type" : "function",
        "function": {
            "name" : "calculate_lead_score",
            "description" : "Calculates a lead score based on the domain info and CRM history",
            "parameters" : {
                "type" : "object",
                "properties" : {
                    "data_summary" : {"type": "string", "description" : "A JSON string containing the domain info and CRM history"},
                },
                "required" : ["data_summary"],
            },
        },
    },

]

In [23]:
def run_agent(user_promt: str)->str:

  print(f"\n--- Running Lead Qualifier Agent ----- ")

  system_prompt = (
      "You are an expert CRM Lead Qualifier Agent. Your sole task is to analyze a sales lead "
      "provided via email address. To do this, you MUST use the available tools. You must follow these steps precisely: "
      "1. Identify the domain from the email. "
      "2. Call `lookup_domain_info` to get company details and `check_crm_history` to get contact history. Execute these calls sequentially. "
      "3. Once both calls are complete, combine all gathered data (domain info and CRM history) into a single JSON object. "
      "4. Call `calculate_lead_score` with this combined JSON object to determine the lead's qualification. "
      "5. Finally, synthesize all information (domain info, CRM history, and the calculated lead score) "
      "into a single, easy-to-read summary for a busy sales rep."
  )
  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_promt},
  ]
  collected_data = {}
  print("Messages sent to OpenAI:", messages) # Added print statement for debugging
  while True:
    print("\n[AI Thinking....]")
    response = client.chat.completions.create(
        model = "gpt-4o",
        messages = messages,
        tools = tools_schema,
        tool_choice = "auto",
    )

    response_message = response.choices[0].message
    messages.append(response_message)

    if response_message.tool_calls:
      tool_calls = response_message.tool_calls

      for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = AVAILABLE_FUNCTIONS.get(function_name)
        function_args = json.loads(tool_call.function.arguments)

        if not function_to_call:
          print(f"Error: Unknown function {function_name}")
          continue

        function_result = function_to_call(**function_args)

        # Correctly store results in collected_data for final summary
        if function_name == "lookup_domain_info":
          collected_data["domain_info"] = json.loads(function_result)
        elif function_name == "check_crm_history":
          collected_data["crm_history"] = json.loads(function_result)
        elif function_name == "calculate_lead_score":
          collected_data["lead_score"] = json.loads(function_result)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": function_name,
            "content": function_result,
        })

    else:
      print("\n -----FINAL AI SUMMARY----")
      print(response_message.content)
      break


In [24]:
# Scenario 1: High-Value Lead (Large company, needs scoring)
lead_email_1 = "dur@acmecorp.com"
run_agent(f"Please qualify this lead for my call tomorrow: {lead_email_1}")

print("\n" + "="*80 + "\n")


--- Running Lead Qualifier Agent ----- 

[AI Thinking....]
-> Tool acticated: Looking for domain info for acmecorp.com...
-> Tool activated: Looking for CRM history for dur@acmecorp.com....

[AI Thinking....]
->Tool activated: Calculating lead score...
{'domain_info': {'industry': 'SAAS', 'size': '500-1000 employees', 'revenue': '$50M-$100M'}, 'crm_history': {'last_contact': '2025-11-15', 'status': 'Cold Lead', 'notes': 'Attended webinar, no follow-up yet.'}}
->Tool activated: Calculating lead score...
{'domain_info': {'industry': 'SAAS', 'size': '500-1000 employees', 'revenue': '$50M-$100M'}, 'crm_history': {'last_contact': '2025-11-15', 'status': 'Cold Lead', 'notes': 'Attended webinar, no follow-up yet.'}}

[AI Thinking....]

 -----FINAL AI SUMMARY----
Here's a summary for your upcoming call with the lead dur@acmecorp.com:

**Company Information:**
- **Industry:** SAAS
- **Size:** 500-1000 employees
- **Revenue:** $50M-$100M

**CRM History:**
- **Last Contact:** November 15, 2025
-

In [25]:
# Scenario 2: Medium-Value Lead (Active opportunity, needs scoring)
lead_email_2 = "bob@widgetco.net"
run_agent(f"Can you run an analysis on this lead: {lead_email_2}")


--- Running Lead Qualifier Agent ----- 

[AI Thinking....]
-> Tool acticated: Looking for domain info for widgetco.net...
-> Tool activated: Looking for CRM history for bob@widgetco.net....

[AI Thinking....]
->Tool activated: Calculating lead score...
{'domain_info': {'industry': 'Unknown', 'size': 'N/A', 'revenue': 'N/A'}, 'crm_history': {'last_contact': 'N/A', 'status': 'No Record', 'notes': 'New lead, first contact opportunity.'}}
->Tool activated: Calculating lead score...
{'domain_info': {'industry': 'Unknown', 'size': 'N/A', 'revenue': 'N/A'}, 'crm_history': {'last_contact': 'N/A', 'status': 'No Record', 'notes': 'New lead, first contact opportunity.'}}

[AI Thinking....]

 -----FINAL AI SUMMARY----
**Lead Analysis Summary for bob@widgetco.net**

- **Domain Information:**
  - **Industry:** Unknown
  - **Company Size:** N/A
  - **Estimated Revenue:** N/A

- **CRM History:**
  - **Last Contact:** N/A
  - **Status:** No Record
  - **Notes:** This is a new lead with the first conta